## Silver Layer
###### banking.bronze.cbs_repayment -> banking.silver.fact_repayment (MERGE)

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
bronze_source= "banking.bronze.cbs_repayment"
silver_target= "banking.silver.fact_repayment"
base="abfss://data@stnajdbanking.dfs.core.windows.net"
checkpoint = f"{base}/silver/fact_repayment/checkpoints/"
key= "event_id"

### Transformation

In [0]:

def transform(df, s):
    # Current dimensions (batch read: we need the full dim, not only new rows)
    customers = s.read.table("banking.silver.dim_customer").select(
        "customer_id", "gender", col("city").alias("customer_city"),
        "customer_segment", "nationality_type")
    branches = s.read.table("banking.silver.dim_branch").select(
        "branch_id", "branch_name", col("city").alias("branch_city"), "region")
    products = s.read.table("banking.silver.dim_product").select(
        "product_code", "product_desc", "product_type", "shariah_structure")

    return (
        df.dropDuplicates([key])
        .withColumn("due_date", to_date("due_date"))
        .withColumn("cure_date", to_date("cure_date"))
        .withColumn("days_past_due", col("days_past_due").cast("int"))
        .withColumn("installment_amount_sar", col("installment_amount_sar").cast("decimal(18,2)"))
        .withColumn("outstanding_balance_sar", col("outstanding_balance_sar").cast("decimal(18,2)"))
        .withColumn("due_month", date_format("due_date", "yyyy-MM"))
        .join(customers, "customer_id", "left")
        .join(branches,  "branch_id",   "left")
        .join(products,  "product_code", "left")
        .withColumn("load_timestamp", current_timestamp())
        .drop("source_file", "bronze_ingested_at")
    )

### Loading into Silver

In [0]:
# Create the empty target once (no-op if it exists), so a later run can never overwrite the fact
transform(spark.read.table(bronze_source).limit(0), spark).write.format("delta").mode("ignore").saveAsTable(silver_target)

In [0]:
def upsert(batch_df, batch_id):
    s = batch_df.sparkSession            # use the batch's session (required on serverless)
    batch_df = transform(batch_df, s)
    if not s.catalog.tableExists(silver_target):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_target)   # first run
    else:
        (DeltaTable.forName(s, silver_target).alias("t")
            .merge(batch_df.alias("s"), f"t.{key} = s.{key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())


In [0]:
(
    spark.readStream.table(bronze_source)
    .writeStream
    .foreachBatch(upsert)
    .outputMode("update")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [0]:
display(spark.table(silver_target).orderBy("event_id"))